# RO-056 State lattices: every number in the Note

The lot and the car are RO-055's (`images/carkit.py`); the lattice, its control set, the cells each move sweeps and its search are in
`images/latticekit.py`. Nothing here is random, so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import carkit as ck, latticekit as lk
from carpaths import step

## 1. Fixed moves never close up

In [2]:
lap = 2 * math.pi * 5
print('one lap', round(lap, 2), 'm =', round(lap / 1.5, 3), 'arcs of 1.5 m')
p = (0.0, -5.0, 0.0); ends = [p]
for _ in range(63):
    p = step(p, 'L', 1.5, 5.0); ends.append(p)
ends = np.array(ends)
print('after 21 arcs the car has turned', round(21 * 0.3, 2), 'rad, a full turn is', round(2 * math.pi, 4), 'rad')
print('end of arc 21 is', round(math.hypot(*(ends[21, :2] - ends[0, :2])), 3), 'm from the start')
gaps = [math.hypot(*(ends[i, :2] - ends[j, :2])) for i in range(64) for j in range(i)]
print('64 end points, smallest gap between two of them', round(min(gaps), 3), 'm')
q = (0.0, -5.0, 0.0)
for _ in range(4):
    q = step(q, 'L', math.pi / 2 * 5, 5.0)
print('four quarter-turn moves of', round(math.pi / 2 * 5, 3), 'm end at', np.round(q[:2], 9))
H0 = ck.hybrid_astar('max', shot=False); H = ck.hybrid_astar('max', shot=True)
e = H0['poses'][-1]
print('Hybrid A* (RO-055): without the shot', H0['expanded'], 'expansions, ends', round(math.hypot(e[0] - 12, e[1] - 23), 2), 'm off;',
      'with the shot', H['expanded'], 'expansions,', round(H['length'], 2), 'm, cost', round(H['cost'], 3))

one lap 31.42 m = 20.944 arcs of 1.5 m
after 21 arcs the car has turned 6.3 rad, a full turn is 6.2832 rad
end of arc 21 is 0.084 m from the start
64 end points, smallest gap between two of them 0.084 m
four quarter-turn moves of 7.854 m end at [-0. -5.]


Hybrid A* (RO-055): without the shot 1949 expansions, ends 0.54 m off; with the shot 312 expansions, 39.42 m, cost 50.835


## 2. A turn that lands on a lattice point

In [3]:
a, b, end = lk.turn_piece(0, 1)
print('arc alone ends at', np.round(step((0, 0, 0), 'L', math.pi / 4 * 5, 5), 4))
print('straight before', round(a, 4), 'm, arc', round(math.pi / 4 * 5, 4), 'm, straight after', round(b, 4), 'm -> ends on', end)
print('length', round(a + math.pi / 4 * 5 + b, 4))
for h in (0, 1):
    print('from heading', 45 * h, 'deg:', len(lk.CONTROL[h]), 'moves')
    for m in lk.CONTROL[h]:
        print('   ', 'forward' if m['d'] > 0 else 'reverse', m['name'], 'ends', m['end'], 'heading change', 45 * m['dh'], 'deg, length', round(m['length'], 3))
print('branching factor by heading:', {45 * h: len(lk.CONTROL[h]) for h in range(8)})

arc alone ends at [3.5355 1.4645 0.7854]
straight before 0.9289 m, arc 3.927 m, straight after 0.7574 m -> ends on (5, 2)
length 5.6133
from heading 0 deg: 10 moves
    forward straight ends (1, 0) heading change 0 deg, length 1.0
    reverse straight ends (-1, 0) heading change 0 deg, length 1.0
    forward left 45 ends (5, 2) heading change 45 deg, length 5.613
    reverse left 45 ends (-5, 2) heading change -45 deg, length 5.613
    forward right 45 ends (5, -2) heading change -45 deg, length 5.613
    reverse right 45 ends (-5, -2) heading change 45 deg, length 5.613
    forward left 90 ends (5, 5) heading change 90 deg, length 7.854
    reverse left 90 ends (-5, 5) heading change -90 deg, length 7.854
    forward right 90 ends (5, -5) heading change -90 deg, length 7.854
    reverse right 90 ends (-5, -5) heading change 90 deg, length 7.854
from heading 45 deg: 6 moves
    forward straight ends (1, 1) heading change 0 deg, length 1.414
    reverse straight ends (-1, -1) heading ch

## 3. Copying a move to a state, and the cells it sweeps

In [4]:
m = next(m for m in lk.CONTROL[0] if m['name'] == 'left 45' and m['d'] > 0)
x, y, th = 17, 9, math.pi / 2
print('the 45 deg left turn (5, 2) copied to (17, 9) facing 90 deg ends at',
      (round(x + math.cos(th) * 5 - math.sin(th) * 2, 6), round(y + math.sin(th) * 5 + math.cos(th) * 2, 6)))
print('precomputed swaths:', len(lk.SWATH), '; cells in each, from', min(len(v) for v in lk.SWATH.values()), 'to', max(len(v) for v in lk.SWATH.values()))
K = next(i for i, m in enumerate(lk.CONTROL[0]) if m['name'] == 'left 90' and m['d'] > 0)
print('swath of the 90 deg left turn:', len(lk.SWATH[(0, K)]), 'cells of 0.1 m; free at (12, 3):', lk.swath_free(12, 3, 0, K))

the 45 deg left turn (5, 2) copied to (17, 9) facing 90 deg ends at (15.0, 14.0)
precomputed swaths: 64 ; cells in each, from 1020 to 3300
swath of the 90 deg left turn: 3260 cells of 0.1 m; free at (12, 3): True


## 4. The search

In [5]:
L = lk.lattice_astar()
print('lattice A*:', L['expanded'], 'expansions; cost', round(L['cost'], 3), '; length', round(L['length'], 3), 'm; switches', L['cusps'])
print('exact body check along the path: free =', L['exact_free'], '; smallest clearance', round(min(ck.clearance(p) for p in L['poses']), 3), 'm')
print('end pose', np.round(L['poses'][-1], 6))
for s, k in L['moves']:
    mv = lk.CONTROL[s[2]][k]
    print('   from', s[:2], 'facing', 45 * s[2], 'deg:', 'forward' if mv['d'] > 0 else 'reverse', mv['name'])
D = lk.lattice_astar(hfun=lambda x, y, h: 0.0)
print('with no guess (Dijkstra):', D['expanded'], 'expansions, cost', round(D['cost'], 4))

Lt = lk.lattice_astar(trace=True)
print('swath look-ups during the search:', sum(len(e['kids']) for e in Lt['log']))
print('Hybrid A* arcs driven during its search:', H['generated'], 'each checked at', int(round(ck.STEP / ck.CHECK_DS)) + 1, 'poses')

lattice A*: 629 expansions; cost 54.789 ; length 42.935 m; switches 1


exact body check along the path: free = True ; smallest clearance 0.25 m
end pose [12.       23.        4.712389]
   from (2, 3) facing 0 deg: forward straight
   from (3, 3) facing 0 deg: forward straight
   from (4, 3) facing 0 deg: forward straight
   from (5, 3) facing 0 deg: forward straight
   from (6, 3) facing 0 deg: forward straight
   from (7, 3) facing 0 deg: forward straight
   from (8, 3) facing 0 deg: forward straight
   from (9, 3) facing 0 deg: forward straight
   from (10, 3) facing 0 deg: forward straight
   from (11, 3) facing 0 deg: forward straight
   from (12, 3) facing 0 deg: forward left 90
   from (17, 8) facing 90 deg: forward straight
   from (17, 9) facing 90 deg: forward left 45
   from (15, 14) facing 135 deg: forward left 45
   from (10, 16) facing 180 deg: forward straight
   from (9, 16) facing 180 deg: forward straight
   from (8, 16) facing 180 deg: forward straight
   from (7, 16) facing 180 deg: reverse right 90
   from (12, 21) facing 270 deg: reve

with no guess (Dijkstra): 1620 expansions, cost 54.7885
swath look-ups during the search: 5192
Hybrid A* arcs driven during its search: 1866 each checked at 7 poses
